<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Munsell_HC_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Munsell Hue/Chroma extractor

This notebook reads an Excel file (e.g. [Kefir.xlsx](https://github.com/postnicov/ResazurinResorufin/raw/refs/heads/main/Data/Kefir.xlsx)) containing a **Munsell code** column (e.g. `9.3P 5.9/3.8`), and adds two numerical columns right after **Munsell comment**:

- **H** — hue mapped onto a single continuous numeric axis, integer part encodes the major hue family in the order `B -> PB -> P -> RP -> R -> YR -> Y -> GY -> G -> BG` (1..10), fractional part is the Munsell hue numerical prefix divided by 10 (e.g. `9.3P` -> hue family `P` = 3, prefix `9.3` -> `3 + 9.3/10 = 3.93`).
- **C** — the Munsell chroma (the number after the `/`).

Neutral codes (`N ...`, no hue/chroma) are left as `NaN` in both columns.

The output file exactly preserves the structure of the input file (cell fills/colors, merged-cell comments/annotations, column widths, and all other columns and rows) except for the addition of the **H** and **C** columns. This is achieved with a custom column-insertion routine, because `openpyxl`'s built-in `insert_cols()` does not correctly shift merged-cell ranges, which corrupts formatting to the right of the insertion point.

The processed file is saved as `<original_name>_HC.xlsx`.


In [8]:
#@title Check and install required packages
import importlib
import subprocess
import sys

required = {
    "openpyxl": "openpyxl",
    "numpy": "numpy",
}

missing = []
for module_name, pip_name in required.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")


All required packages are already installed.


In [9]:
#@title Import libraries
import re
import os
import numpy as np
import openpyxl
from openpyxl.utils import get_column_letter, range_boundaries, column_index_from_string
from copy import copy
from google.colab import files


In [10]:
#@title Upload the Excel file
uploaded = files.upload()
input_filename = next(iter(uploaded.keys()))
print(f"Uploaded file: {input_filename}")


Saving Kefir.xlsx to Kefir.xlsx
Uploaded file: Kefir.xlsx


In [11]:
#@title Define the Munsell hue -> numeric mapping and parsing logic
# Munsell major-hue sequence used for the integer part of H,
# following the order requested: B -> PB -> P -> RP -> R -> YR -> Y -> GY -> G -> BG (1..10)
HUE_ORDER = ["B", "PB", "P", "RP", "R", "YR", "Y", "GY", "G", "BG"]
HUE_TO_INT = {hue: i + 1 for i, hue in enumerate(HUE_ORDER)}

# Matches codes like "9.3P 5.9/3.8", "9.1RP 5.9/2.9", "2.3R 6.0/3.3"
MUNSELL_RE = re.compile(
    r"^\s*([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z]+)\s+([0-9]+(?:\.[0-9]+)?)\s*/\s*([0-9]+(?:\.[0-9]+)?)\s*$"
)

# Matches neutral codes like "N 61.9" (no hue, no chroma)
NEUTRAL_RE = re.compile(r"^\s*N\s*[0-9]+(?:\.[0-9]+)?\s*$")


def parse_munsell_code(code):
    """Return (H, C) for a Munsell code string, or (NaN, NaN) if it cannot be parsed
    (including neutral 'N ...' codes, which have no hue/chroma)."""
    if code is None:
        return np.nan, np.nan

    code = str(code).strip()
    if not code or code.lower() == "nan":
        return np.nan, np.nan

    if NEUTRAL_RE.match(code):
        return np.nan, np.nan

    m = MUNSELL_RE.match(code)
    if not m:
        print(f"Warning: could not parse Munsell code '{code}'")
        return np.nan, np.nan

    prefix_str, hue_letters, _value_str, chroma_str = m.groups()
    hue_letters = hue_letters.upper()

    if hue_letters not in HUE_TO_INT:
        print(f"Warning: unknown hue family '{hue_letters}' in code '{code}'")
        return np.nan, np.nan

    prefix = float(prefix_str)
    hue_int = HUE_TO_INT[hue_letters]
    H = hue_int + prefix / 10.0
    C = float(chroma_str)
    return H, C


In [12]:
#@title Structure-preserving column insertion helper
def insert_columns_safely(ws, insert_at, amount):
    """Insert `amount` blank columns before column index `insert_at` (1-based),
    shifting cell values, styles/fills, hyperlinks, comments, merged-cell ranges,
    and column widths accordingly. Unlike openpyxl\'s ws.insert_cols(), this keeps
    merged-cell ranges (e.g. section-header comments) correctly aligned, so the
    rest of the sheet\'s structure and formatting is left untouched."""
    max_col = ws.max_column
    max_row = ws.max_row

    # Remember and unmerge all merged ranges first (avoids read-only MergedCell errors)
    old_ranges = [str(r) for r in ws.merged_cells.ranges]
    for r in old_ranges:
        ws.unmerge_cells(r)

    # Shift cell values + styles for columns >= insert_at rightward by `amount`
    # (iterate right-to-left so we never overwrite a column before reading it)
    for col in range(max_col, insert_at - 1, -1):
        for row in range(1, max_row + 1):
            src = ws.cell(row=row, column=col)
            dst = ws.cell(row=row, column=col + amount)
            dst.value = src.value
            dst._style = copy(src._style)
            if src.hyperlink:
                dst.hyperlink = copy(src.hyperlink)
            if src.comment:
                dst.comment = copy(src.comment)

    # Clear the newly freed columns (insert_at .. insert_at + amount - 1)
    blank_style = ws.cell(row=1, column=1)._style.__class__()
    for col in range(insert_at, insert_at + amount):
        for row in range(1, max_row + 1):
            cell = ws.cell(row=row, column=col)
            cell.value = None
            cell._style = copy(blank_style)

    # Recompute and re-apply merged ranges with shifted columns
    for r in old_ranges:
        min_col, min_row, max_col_r, max_row_r = range_boundaries(r)
        if min_col >= insert_at:
            new_min_col, new_max_col = min_col + amount, max_col_r + amount
        elif max_col_r >= insert_at:
            new_min_col, new_max_col = min_col, max_col_r + amount
        else:
            new_min_col, new_max_col = min_col, max_col_r
        new_range = f"{get_column_letter(new_min_col)}{min_row}:{get_column_letter(new_max_col)}{max_row_r}"
        ws.merge_cells(new_range)

    # Shift column widths (column_dimensions is keyed by column letter)
    dims = dict(ws.column_dimensions)
    new_dims = {}
    for col_letter, dim in dims.items():
        col_idx = column_index_from_string(col_letter)
        new_letter = get_column_letter(col_idx + amount) if col_idx >= insert_at else col_letter
        new_dims[new_letter] = dim
    for letter in list(ws.column_dimensions.keys()):
        del ws.column_dimensions[letter]
    for letter, dim in new_dims.items():
        dim.index = letter
        ws.column_dimensions[letter] = dim


In [13]:
#@title Compute H and C, and insert them after \'Munsell comment\' (structure fully preserved)
wb = openpyxl.load_workbook(input_filename)
ws = wb.active

header = [cell.value for cell in ws[1]]

if "Munsell code" not in header:
    raise ValueError("Column \'Munsell code\' not found in the input file.")
if "Munsell comment" not in header:
    raise ValueError("Column \'Munsell comment\' not found in the input file.")

col_munsell = header.index("Munsell code") + 1
col_comment = header.index("Munsell comment") + 1

# Compute H, C for every data row before touching the sheet layout
hc_values = []
for row_idx in range(2, ws.max_row + 1):
    code_value = ws.cell(row=row_idx, column=col_munsell).value
    hc_values.append(parse_munsell_code(code_value))

insert_at = col_comment + 1
insert_columns_safely(ws, insert_at, 2)

ws.cell(row=1, column=insert_at, value="H")
ws.cell(row=1, column=insert_at + 1, value="C")

for offset, (H, C) in enumerate(hc_values):
    row_idx = offset + 2
    ws.cell(row=row_idx, column=insert_at, value=None if np.isnan(H) else H)
    ws.cell(row=row_idx, column=insert_at + 1, value=None if np.isnan(C) else C)

print("H and C columns inserted after \'Munsell comment\'; rest of the sheet is unchanged.")


H and C columns inserted after 'Munsell comment'; rest of the sheet is unchanged.


In [14]:
#@title Export the processed file and download it
base_name, ext = os.path.splitext(input_filename)
output_filename = f"{base_name}_HC{ext if ext else '.xlsx'}"

wb.save(output_filename)
print(f"Saved processed file as: {output_filename}")

files.download(output_filename)


Saved processed file as: Kefir_HC.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>